# 🗄️ Lesson 09: Vector Databases

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/blob/main/notebooks/09_vector_databases.ipynb)

> **A Python variable forgets everything when you close the notebook. A vector database remembers, searches fast, and filters.**

---

## 🎯 Goal
By the end of this lesson you can store embeddings with their text and metadata, search them, filter by metadata, and keep them on disk.

## ℹ️ Lesson info

| | |
|---|---|
| ⏱ **Time** | ~25 minutes |
| 📚 **Prerequisites** | [Lesson 08](08_choosing_embedding_model.ipynb) |
| 🔑 **API key** | Not needed |
| 🛠 **Libraries** | `chromadb`, `faiss-cpu`, `sentence-transformers`, `numpy` |

In [ ]:
!pip install -q -U chromadb faiss-cpu sentence-transformers numpy

## 🤔 Why not just keep vectors in a numpy array?

That is what we did in Lessons 07 and 08. It works for 4 chunks, but:

| Problem with a plain array | What a vector database gives |
|---|---|
| Disappears when the notebook closes | **Persistence** on disk |
| Compares the question with *every* vector | **Approximate search** that stays fast with millions of vectors |
| No place for text and metadata | Stores **vector + text + metadata** together |
| Filtering means writing loops | **Filters** like "only section = PTO" |
| Hard to add, change or delete items | `add`, `upsert`, `delete` |

In this lesson we use **Chroma**: free, runs locally, and needs no server.

In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBED_MODEL = "intfloat/multilingual-e5-small"
embedder = SentenceTransformer(EMBED_MODEL)

chunks = [
    {"id": "pdf1-0", "section": "NDA", "text": "Employees must protect confidential information belonging to the company, its clients and partners, such as product roadmaps, customer data and proprietary algorithms. It must not be shared with unauthorized people."},
    {"id": "pdf1-1", "section": "Conduct", "text": "Employees must maintain a respectful, professional environment free from harassment, discrimination and intimidation. Disruptive behavior and verbal abuse are prohibited and may lead to disciplinary action."},
    {"id": "pdf1-2", "section": "PTO", "text": "Full-time employees receive paid time off: 10 days per year with under 1 year of service, 15 days with 1 to 3 years, and 20 days with more than 3 years. Requests go through the HR system."},
    {"id": "pdf1-3", "section": "Travel", "text": "Reasonable and necessary expenses of approved business trips are reimbursed. Flights, hotels and meals must be recorded with receipts in the finance system."},
]

for chunk in chunks:
    chunk["metadata"] = {
        "source": "pdf1.pdf",
        "section": chunk["section"],
        "page": 0,
        "word_count": len(chunk["text"].split()),
    }

chunk_vectors = embedder.encode(["passage: " + chunk["text"] for chunk in chunks], normalize_embeddings=True)
print("vectors:", chunk_vectors.shape)

d:\AI Source\Rag_torurial\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1604.46it/s]


vectors: (4, 384)


## 🧱 Step 1: Create a collection and store the chunks

In Chroma, a **collection** is like a table. Each item has:

| Field | What it is |
|---|---|
| `id` | a unique string |
| `embedding` | the vector |
| `document` | the original text (so we can show it later) |
| `metadata` | a dictionary with extra info (`section`, `page`, ...) |

Two choices matter here:
- **`PersistentClient(path=...)`** keeps the data on disk in that folder.
- **`space: cosine`** makes Chroma measure closeness with cosine, like we did by hand.
- We use **`upsert`** (insert-or-update), so re-running the cell doesn't create duplicates.

We pass our **own embeddings**, so Chroma doesn't use its built-in model.

In [2]:
import chromadb

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_or_create_collection(
    name="handbook",
    configuration={"hnsw": {"space": "cosine"}},
)

collection.upsert(
    ids=[chunk["id"] for chunk in chunks],
    embeddings=chunk_vectors.tolist(),
    documents=[chunk["text"] for chunk in chunks],
    metadatas=[chunk["metadata"] for chunk in chunks],
)

print("stored:", collection.count())

stored: 4


## 🔍 Step 2: Search

Chroma returns a **distance**, not a similarity. For cosine space:

`distance = 1 - cosine similarity`

So **smaller distance means more similar**. A distance of `0` means identical direction. We print both to avoid confusion.

In [3]:
def search(question, top_k=2, where=None):
    question_vector = embedder.encode(["query: " + question], normalize_embeddings=True)[0]
    result = collection.query(
        query_embeddings=[question_vector.tolist()],
        n_results=top_k,
        where=where,
    )
    return list(zip(result["distances"][0], result["metadatas"][0], result["documents"][0]))

for distance, metadata, text in search("How many vacation days do I get after two years?", top_k=4):
    print(f'distance={distance:.3f}  similarity={1 - distance:.3f}  | {metadata["section"]}')

distance=0.151  similarity=0.849  | PTO
distance=0.207  similarity=0.793  | Travel
distance=0.216  similarity=0.784  | Conduct
distance=0.230  similarity=0.770  | NDA


### 👀 What we saw

| Rank | Section | Similarity |
|---|---|---|
| 1 | PTO | 0.849 |
| 2 | Travel | 0.793 |
| 3 | Conduct | 0.784 |
| 4 | NDA | 0.770 |

PTO is first, as it should be. But notice how **close the scores are**: even the *worst* chunk scores 0.77. This model gives high similarity to everything, so a score like "0.8" doesn't mean "relevant". The **order** is informative, the absolute number much less. (This is one question and one model: don't generalize.)

## 🎯 Step 3: Filter by metadata

Now the metadata we attached in Lesson 05 pays off. The `where` argument tells Chroma to search **only among items that match**.

| Filter | Meaning |
|---|---|
| `{"section": "Travel"}` | exact match |
| `{"word_count": {"$gt": 28}}` | greater than 28 (also `$gte`, `$lt`, `$lte`, `$ne`) |
| `{"$and": [ ... , ... ]}` | all conditions must hold (also `$or`) |

In [4]:
question = "How many vacation days do I get after two years?"

print("only section = Travel:")
for distance, metadata, text in search(question, top_k=4, where={"section": "Travel"}):
    print(f'   similarity={1 - distance:.3f} | {metadata["section"]}')

print("word_count > 28:")
for distance, metadata, text in search(question, top_k=4, where={"word_count": {"$gt": 28}}):
    print(f'   similarity={1 - distance:.3f} | {metadata["section"]} ({metadata["word_count"]} words)')

print("word_count > 28 AND section = NDA:")
for distance, metadata, text in search(
    question, top_k=4,
    where={"$and": [{"word_count": {"$gt": 28}}, {"section": "NDA"}]},
):
    print(f'   similarity={1 - distance:.3f} | {metadata["section"]}')

only section = Travel:
   similarity=0.793 | Travel
word_count > 28:
   similarity=0.849 | PTO (37 words)
   similarity=0.770 | NDA (30 words)
word_count > 28 AND section = NDA:
   similarity=0.770 | NDA


## 💾 Step 4: Persistence, update and delete

Because we used `PersistentClient(path="chroma_db")`, the data lives in the folder `chroma_db/`. If you close the notebook and open it tomorrow, the collection is still there. We'll simulate that by opening a **second client** on the same folder.

We'll also:
- **update** the metadata of one item (`reviewed: True`),
- **delete** one item and put it back with `upsert`.

> ⚠️ Add `chroma_db/` to `.gitignore`: it is generated data, not source code.

In [5]:
# 1) reopen the database from disk, as if the notebook had been closed
reopened_client = chromadb.PersistentClient(path="chroma_db")
reopened_collection = reopened_client.get_collection("handbook")
print("items after reopening:", reopened_collection.count())

# 2) update the metadata of one item
collection.update(ids=["pdf1-2"], metadatas=[{**chunks[2]["metadata"], "reviewed": True}])
print(collection.get(ids=["pdf1-2"])["metadatas"][0])

# 3) delete one item, then put it back
collection.delete(ids=["pdf1-3"])
print("after delete:", collection.count())

collection.upsert(
    ids=[chunks[3]["id"]],
    embeddings=[chunk_vectors[3].tolist()],
    documents=[chunks[3]["text"]],
    metadatas=[chunks[3]["metadata"]],
)
print("after upsert:", collection.count())

items after reopening: 4
{'source': 'pdf1.pdf', 'word_count': 37, 'page': 0, 'section': 'PTO', 'reviewed': True}
after delete: 3
after upsert: 4


## ⚡ A second tool: FAISS

**FAISS** (from Meta) is a library, not a database. It does one thing very well: **fast similarity search over vectors.**

| | Chroma | FAISS |
|---|---|---|
| Stores vectors | yes | yes |
| Stores text and metadata | **yes** | **no** (you keep them yourself) |
| Filtering by metadata | **yes** (`where`) | no, not directly |
| Saves to disk | automatically | you call `write_index` yourself |
| Typical use | an app that needs everything in one place | raw speed, or a building block inside a bigger system |

FAISS only gives back **positions** (0, 1, 2, ...). It's *your* job to map position `2` back to the chunk. That is why we keep the `chunks` list in the same order as the vectors.

We use `IndexFlatIP`: an **exact** search (it compares with every vector) using the **inner product**. With normalized vectors, inner product = cosine similarity, and **higher is better** (no distance to convert).

In [6]:
import faiss

index = faiss.IndexFlatIP(chunk_vectors.shape[1])
index.add(chunk_vectors.astype("float32"))
print("vectors in index:", index.ntotal)

question = "How many vacation days do I get after two years?"
question_vector = embedder.encode(["query: " + question], normalize_embeddings=True).astype("float32")

similarities, positions = index.search(question_vector, 4)

for similarity, position in zip(similarities[0], positions[0]):
    print(f'similarity={similarity:.3f}  position={position}  | {chunks[position]["section"]}')

# FAISS saves to disk only when you ask
faiss.write_index(index, "handbook.faiss")
loaded_index = faiss.read_index("handbook.faiss")
print("loaded from disk:", loaded_index.ntotal)

vectors in index: 4
similarity=0.849  position=2  | PTO
similarity=0.793  position=3  | Travel
similarity=0.784  position=1  | Conduct
similarity=0.770  position=0  | NDA
loaded from disk: 4


### 👀 What we saw

| Rank | Chroma | FAISS |
|---|---|---|
| 1 | PTO (0.849) | PTO (0.849) |
| 2 | Travel (0.793) | Travel (0.793) |
| 3 | Conduct (0.784) | Conduct (0.784) |
| 4 | NDA (0.770) | NDA (0.770) |

Same vectors, same ranking, same scores. The difference is **what comes with the search**: Chroma also gave us the text and metadata, and let us filter. With FAISS we had to map `position=2` back to `chunks[2]` ourselves.

> ℹ️ With only 4 vectors, an exact search and an approximate one give the same answer. The difference appears with millions of vectors (Lessons 10 and 33).

## ⚠️ Things to watch out for

- **Metadata types are limited.** In the Chroma version we used (1.5.9), text, numbers, `True`/`False` and lists were accepted, but a **nested dictionary** raised a `ValueError`. Other versions may differ, so check yours.
- **Distance is not similarity.** Chroma returns `distance = 1 - similarity` for cosine. Smaller is better. FAISS (inner product) returns the similarity itself. Larger is better. Mixing them up flips your ranking.
- **One model per collection.** Embed chunks and questions with the **same model**. If you change the model, rebuild the whole collection.
- **Scores are not a relevance meter.** Here even the worst chunk scored 0.77. Use the **order**, and decide any threshold by measuring.
- **Ids must be unique and stable.** We used ids like `pdf1-2`. With `upsert`, the same id means "replace", so re-running a cell doesn't create duplicates.
- **Generated files don't belong in Git.** Add `chroma_db/` and `*.faiss` to `.gitignore`.

## 📌 Summary

- A **vector database** stores vectors together with their **text and metadata**, searches them quickly, filters, and **persists** to disk.
- In **Chroma** we created a collection (`cosine` space), stored chunks with `upsert`, searched with `query`, filtered with `where` (`$gt`, `$and`, ...), and updated and deleted items.
- Chroma returns a **distance**: `similarity = 1 - distance`.
- **FAISS** is a fast search *library*: it returns only positions and scores, and you keep the text and metadata yourself.
- Both gave the **same ranking** on our data. Choose by the features you need, not by this result.
- Always use the same embedding model for storing and searching.

## ✍️ Exercise (10 minutes)

1. Add a fifth chunk about **"Equipment"** (for example: *"The company provides one laptop per employee. Lost or broken equipment must be reported to IT within 24 hours."*) to `chunks`, with its own `id` and `metadata`.
2. Embed it and add it to the collection with `upsert`.
3. Ask *"My laptop broke, what should I do?"* with `search`. Is the new chunk first?

**Success:** the new chunk is found, and `collection.count()` is 5.

💡 Solution: `solutions/09_vector_databases.ipynb` (try on your own first)

## 🔥 Challenge (optional)

Our model is multilingual. Store **both** the English chunks and the Persian chunks from Lesson 08 in one collection, adding a metadata field `language` (`"en"` or `"fa"`).

1. Ask the *Persian* question `بعد از دو سال کار چند روز مرخصی دارم؟` with no filter. Which chunks come first: English or Persian?
2. Repeat with `where={"language": "en"}`. Does the multilingual model still find the PTO chunk?

What does this tell you about searching across languages?

## 📖 Next

We trusted the database to find "the closest" vectors, and to do it fast. How does it really measure closeness, and how can it search millions of vectors without comparing all of them? That's the next lesson.

**➡️ Next lesson:** [10 Similarity & Indexing](10_similarity_and_indexing.ipynb)